In [ ]:
import pandas as pd
import numpy as np

In [ ]:
date_rng = pd.date_range(start='1/1/2023', end='6/30/2023', freq='h')
df = pd.DataFrame(date_rng, columns=['timestamp'])

In [ ]:
df['fridge'] = np.random.randint(50, 200, len(df)) * np.random.choice([0,1], len(df), p=[0.3,0.7])

In [ ]:
month = df['timestamp'].dt.month
ac_prob = np.where(month >= 4, 0.8, 0.1)
df['ac'] = np.random.randint(1000, 2500, len(df)) * np.random.binomial(1, ac_prob)

In [ ]:
hour = df['timestamp'].dt.hour
light_prob = np.where((hour >= 18) | (hour <= 6), 0.9, 0.1)

df['lights'] = np.random.randint(20, 100, len(df)) * np.random.binomial(1, light_prob)
df['microwave'] = np.random.randint(800, 1200, len(df)) * np.random.choice([0,1], len(df), p=[0.9,0.1])

In [ ]:
df['total_power'] = df[['fridge','ac','lights','microwave']].sum(axis=1)
df.head()

In [ ]:
#analysis.ipynb

In [ ]:
df['timestamp'] = pd.to_datetime(df['timestamp'])

In [ ]:
#data understanding

In [ ]:
df.info()
df.describe()

In [ ]:
## Trend Analysis

In [ ]:
import matplotlib.pyplot as plt

df['total_power'].plot(figsize=(10,5))
plt.title("Total Energy Consumption Over Time")
plt.show()

In [ ]:
## Device Comparison

In [ ]:
df[['fridge','ac','lights','microwave']].mean().plot(kind='bar')
plt.title("Average Power per Device")
plt.show()

In [ ]:
## Time Analysis

In [ ]:
df['hour'] = df['timestamp'].dt.hour

df.groupby('hour')['total_power'].mean().plot(marker='o')
plt.title("Average Power by Hour")
plt.show()

In [ ]:
## Relationship Analysis

In [ ]:
import seaborn as sns

sns.heatmap(df[['fridge','ac','lights','microwave','total_power']].corr(), annot=True)
plt.title("Correlation Between Devices")
plt.show()

In [ ]:
## Time Series Decomposition

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

# Set timestamp as index
df.set_index('timestamp', inplace=True)

# Decomposition (daily pattern → 24 hours)
decomposition = seasonal_decompose(df['total_power'], model='additive', period=24)

# Plot
decomposition.plot()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose

# Decomposition using index
result = seasonal_decompose(df['total_power'], model='additive', period=24)

result.plot()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Make sure timestamp is index
df = df.copy()
if 'timestamp' in df.columns:
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df.set_index('timestamp', inplace=True)

# 1. Trend (using rolling mean)
df['trend'] = df['total_power'].rolling(window=24).mean()

# 2. Detrended (remove trend)
df['detrended'] = df['total_power'] - df['trend']

# 3. Plot everything
plt.figure(figsize=(12,8))

plt.subplot(3,1,1)
plt.plot(df['total_power'])
plt.title("Original Data")

plt.subplot(3,1,2)
plt.plot(df['trend'])
plt.title("Trend (Rolling Mean)")

plt.subplot(3,1,3)
plt.plot(df['detrended'])
plt.title("Detrended Data")

plt.tight_layout()
plt.show()

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

plot_acf(df['total_power'], lags=50)
plt.title("Autocorrelation of Total Power")
plt.show()

In [ ]:
## Partial Autocorrelation (PACF)

In [ ]:
from statsmodels.graphics.tsaplots import plot_pacf

plot_pacf(df['total_power'], lags=50)
plt.title("Partial Autocorrelation")
plt.show()

In [ ]:
#hourly.ipynb

In [ ]:
df['hour'] = df.index.hour
df.head()

In [ ]:
##hourly table

In [ ]:
hourly_table = df.groupby('hour').agg({
    'total_power': ['mean','max','min'],
    'fridge': 'mean',
    'ac': 'mean',
    'lights': 'mean',
    'microwave': 'mean'
})

hourly_table

In [ ]:
##average power by hour

In [ ]:
import matplotlib.pyplot as plt

df.groupby('hour')['total_power'].mean().plot(marker='o')
plt.title("Average Power by Hour")
plt.show()

In [ ]:
##maximum power by hour

In [ ]:
df.groupby('hour')['total_power'].max().plot(kind='bar')
plt.title("Maximum Power by Hour")
plt.show()

In [ ]:
##minimum power by hour

In [ ]:
df.groupby('hour')['total_power'].min().plot(kind='bar')
plt.title("Minimum Power by Hour")
plt.show()

In [ ]:
##Device-wise Hourly Usage

In [ ]:
df.groupby('hour')[['fridge','ac','lights','microwave']].mean().plot()
plt.title("Device-wise Hourly Usage")
plt.show()

In [ ]:
##Peak Hour Detection

In [ ]:
peak_hour = df.groupby('hour')['total_power'].mean().idxmax()
print("Peak usage hour:", peak_hour)

In [ ]:
##weekly.ipynb

In [ ]:
df['day_type'] = df.index.dayofweek.map(lambda x: 'Weekend' if x >= 5 else 'Weekday')
df.head()

In [ ]:
##weeday vs weekend

In [ ]:
daytype_table = df.groupby('day_type').agg({
    'total_power': ['mean','max','min'],
    'fridge': 'mean',
    'ac': 'mean',
    'lights': 'mean',
    'microwave': 'mean'
})

daytype_table

In [ ]:
##Compare Total Power

In [ ]:
import matplotlib.pyplot as plt

df.groupby('day_type')['total_power'].mean().plot(kind='bar')
plt.title("Weekday vs Weekend Energy Usage")
plt.show()

In [ ]:
##Hourly Pattern for Weekday vs Weekend

In [ ]:
df['hour'] = df.index.hour

df.groupby(['hour','day_type'])['total_power'].mean().unstack().plot()
plt.title("Hourly Usage: Weekday vs Weekend")
plt.show()

In [ ]:
##Device-wise Comparison

In [ ]:
df.groupby('day_type')[['fridge','ac','lights','microwave']].mean().plot(kind='bar')
plt.title("Device Usage: Weekday vs Weekend")
plt.show()

In [ ]:
##Peak Time (Weekday vs Weekend) 

In [ ]:
peak = df.groupby(['hour','day_type'])['total_power'].mean().reset_index()

weekday_peak = peak[peak['day_type']=='Weekday'].sort_values('total_power', ascending=False).iloc[0]
weekend_peak = peak[peak['day_type']=='Weekend'].sort_values('total_power', ascending=False).iloc[0]

print("Weekday Peak Hour:", weekday_peak['hour'])
print("Weekend Peak Hour:", weekend_peak['hour'])

In [ ]:
##Energy Difference (Insight Program)

In [ ]:
weekday_avg = df[df['day_type']=='Weekday']['total_power'].mean()
weekend_avg = df[df['day_type']=='Weekend']['total_power'].mean()

print("Weekday Avg:", weekday_avg)
print("Weekend Avg:", weekend_avg)
print("Difference:", weekend_avg - weekday_avg)

In [ ]:
##monthly.ipynb

In [ ]:
df['month'] = df.index.month
df['month_name'] = df.index.month_name()
df.head()

In [ ]:
##Monthly Summary Table 

In [ ]:
monthly_table = df.groupby('month_name').agg({
    'total_power': ['mean','max','min'],
    'fridge': 'mean',
    'ac': 'mean',
    'lights': 'mean',
    'microwave': 'mean'
})

monthly_table

In [ ]:
##Total Power by Month

In [ ]:
import matplotlib.pyplot as plt

df.groupby('month_name')['total_power'].mean().plot(kind='bar')
plt.title("Average Power by Month")
plt.show()

In [ ]:
##Compare Few Months

In [ ]:
selected_months = ['January','February','March']

df[df['month_name'].isin(selected_months)] \
.groupby('month_name')['total_power'].mean() \
.plot(kind='bar')

plt.title("Comparison of First 3 Months")
plt.show()

In [ ]:
##Device Usage for Few Months 

In [ ]:
df[df['month_name'].isin(['April','May','June'])] \
.groupby('month_name')[['ac','fridge','lights','microwave']] \
.mean().plot(kind='bar')

plt.title("Device Usage (Summer Months)")
plt.show()

In [ ]:
##Monthly Peak Detection

In [ ]:
peak_month = df.groupby('month_name')['total_power'].mean().idxmax()
print("Peak usage month:", peak_month)

In [ ]:
##Monthly Trend Line 

In [ ]:
df.groupby('month')['total_power'].mean().plot(marker='o')
plt.title("Monthly Trend of Energy Usage")
plt.show()

In [ ]:
##Winter vs Summer comparison

In [ ]:
winter = ['January','February','March']
summer = ['April','May','June']

print("Winter Avg:", df[df['month_name'].isin(winter)]['total_power'].mean())
print("Summer Avg:", df[df['month_name'].isin(summer)]['total_power'].mean())

In [ ]:
## AC usage increase

In [ ]:
df.groupby('month_name')['ac'].mean().plot(kind='bar')
plt.title("AC Usage Across Months")
plt.show()

In [ ]:
##polynomial.ipynb

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(-10, 10, 100)
y = x**2

plt.plot(x, y)
plt.title("Polynomial Curve: y = x^2")
plt.show()

In [ ]:
##multiple polynomial curves

In [ ]:
x = np.linspace(-10, 10, 100)

plt.plot(x, x**2, label="x^2")
plt.plot(x, x**3, label="x^3")
plt.plot(x, x**4, label="x^4")

plt.legend()
plt.title("Multiple Polynomial Curves")
plt.show()

In [ ]:
##polynomial regression 

In [ ]:
# sample data
x = np.array([1,2,3,4,5])
y = np.array([2,4,5,4,5])

# fit polynomial (degree 2)
coeff = np.polyfit(x, y, 2)
poly = np.poly1d(coeff)

# plot
x_new = np.linspace(1,5,100)
plt.scatter(x, y)
plt.plot(x_new, poly(x_new))
plt.title("Polynomial Regression (Degree 2)")
plt.show()

In [ ]:
##compare linear vs polynomial regression

In [ ]:
# linear fit
linear = np.polyfit(x, y, 1)
linear_fn = np.poly1d(linear)

# polynomial fit
poly = np.polyfit(x, y, 3)
poly_fn = np.poly1d(poly)

plt.scatter(x, y)
plt.plot(x_new, linear_fn(x_new), label="Linear")
plt.plot(x_new, poly_fn(x_new), label="Polynomial")

plt.legend()
plt.title("Linear vs Polynomial")
plt.show()

In [ ]:
prediction.ipynb

In [ ]:
# Predict energy trend using polynomial
x = np.arange(len(df))
y = df['total_power']

coeff = np.polyfit(x, y, 2)
poly = np.poly1d(coeff)

plt.plot(x, y, label="Actual")
plt.plot(x, poly(x), label="Predicted")

plt.legend()
plt.show()